

**The Hybrid Equation: KNOWN PHYSICS + UDE**
$$ C_m \frac{dV}{dt} = I_{ext} - \mathbf{NN(V)} - I_{K}(V) - I_{L}(V) $$


In [30]:
using SciMLSensitivity
using DifferentialEquations
using SciMLSensitivity   # or DiffEqSensitivity if you prefer
using Zygote
using Optimisers         # for optimizer & update
using LinearAlgebra
using DifferentialEquations
using Flux
using Plots
using Optimization
using OptimizationOptimisers
using Zygote
using DataFrames
using  JLD2
using Random
Random.seed!(1234)
println("All the nessecessary packages have been imported")

All the nessecessary packages have been imported


In [31]:
@load "Data/synthetic_data/noise_0_hh_2d_model.jld"

3-element Vector{Symbol}:
 :t
 :V
 :n

In [32]:
# Physics Constants (Explicit Float32)
const C_m  = 1.0f0      # µF/cm²
const g_Na = 120.0f0    # mS/cm²
const E_Na = 50.0f0     # mV
const g_L  = 0.3f0      # mS/cm²
const E_L  = -54.387f0  # mV

-54.387f0

In [33]:
# Voltage-gated ion channel kinetics (Float32)

α_n(V::Float32) = 0.01f0 * (V + 55f0) / (1f0 - exp(-(V + 55f0) / 10f0))
β_n(V::Float32) = 0.125f0 * exp(-(V + 65f0) / 80f0)

α_m(V::Float32) = 0.1f0 * (V + 40f0) / (1f0 - exp(-(V + 40f0) / 10f0))
β_m(V::Float32) = 4.0f0 * exp(-(V + 65f0) / 18f0)

α_h(V::Float32) = 0.07f0 * exp(-(V + 65f0) / 20f0)
β_h(V::Float32) = 1f0 / (1f0 + exp(-(V + 35f0) / 10f0))

# Steady-state & time-constant functions (Float32)

m_inf(V::Float32) = α_m(V) / (α_m(V) + β_m(V))
h_inf(V::Float32) = α_h(V) / (α_h(V) + β_h(V))
n_inf(V::Float32) = α_n(V) / (α_n(V) + β_n(V))

tau_n(V::Float32) = 1f0 / (α_n(V) + β_n(V))

println("Physics of neural dynamics defined in Float32")


Physics of neural dynamics defined in Float32


In [34]:
NN_Model = Chain(
    Dense(1, 32 , tanh), # Inputs: V and n
    Dense(32 , 16, tanh),
    Dense(16, 1)
)|> Flux.f32

Chain(
  Dense(1 => 32, tanh),                 # 64 parameters
  Dense(32 => 16, tanh),                # 528 parameters
  Dense(16 => 1),                       # 17 parameters
)                   # Total: 6 arrays, 609 parameters, 2.684 KiB.

In [35]:
p_nn, re = Flux.destructure(NN_Model)
p_nn = Float32.(p_nn)
println("Recruit Constructed. Parameters: ", length(p_nn))
println("Parameter element type: ", eltype(p_nn))


Recruit Constructed. Parameters: 609
Parameter element type: Float32


In [36]:
function Stimulus(t)
    # A 1ms pulse starting at 10ms
    return(t>=10.0 && t<11.0) ? 20 : 0.0
end

println(" An extra current form neighbour to generate a pulse")

 An extra current form neighbour to generate a pulse


In [37]:
using  Statistics
const V_mean  = mean(V)
const V_std = std(V)
norm_input(v)=(v-V_mean)/V_std



norm_input (generic function with 1 method)

In [39]:
function hodgkin_huxley_UDE!(du,u,p,t)
          V,n = u 



          nn_model = re(p)
          prob_I_Na = nn_model([norm_input(V)])[1]
          I_ext = Stimulus(t)
          I_K   = g_K * n^4 * (V - E_K)
          I_L   = g_L * (V - E_L)
          # Dynamics
    du[1] = (I_ext - pred_I_Na - I_K - I_L) / Cm
    du[2] = (n_inf(V) - n) / tau_n(V)
end

# Initial condition (Float64 vector)
u0_true = [-65.0f0, n_inf(-65.0f0)]

# Time span (Float64)
t_span = (0.0f0, 50.0f0)

prob_nn = ODEProblem(hodgkin_huxley_UDE! , u0_true,t_span , p_nn)

ODEProblem with uType Vector{Float32} and tType Float32. In-place: true
Non-trivial mass matrix: false
timespan: (0.0f0, 50.0f0)
u0: 2-element Vector{Float32}:
 -65.0
   0.3176769

In [ ]:
# 

function predeict_ude(p)
    

UndefVarError: UndefVarError: `prob_ture` not defined in `Main`
Suggestion: check for spelling errors or missing imports.